# 04 - PLM embeddings (Rep 2: mean-pooled, site, site-delta)

Pulls three embedding slices out of the same forward pass for each PLM, so nothing gets re-run per representation. Rep 2a is mean-pooled: the average over all 263 residues, a whole-sequence summary. Rep 2b is the site embedding, just the mutated position, with no dilution from the rest of the sequence. Rep 2c is the site-delta: mutant minus WT at that same site, isolating the effect of the substitution itself.

About 4,770 forward passes per model, so this is the expensive tier. Shells to `esm` / `esmc`.

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import subprocess, sys
def run_in(env, script, *args):
    """Run src/<script> in conda env <env>; stream output; raise on failure."""
    cmd = ["conda","run","--no-capture-output","-n",env,"python",str(SRC/script), *map(str,args)]
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0: raise RuntimeError(f"{script} failed in env {env} (rc={r.returncode})")


In [ ]:
for tag,(env,model,layer) in {
    'esm2_650m':('esm','esm2_t33_650M_UR50D',33),
    'esm1v':('esm','esm1v_t33_650M_UR90S_1',33),
}.items():
    if (INTERIM/f'emb3_{tag}.npz').exists(): print('cached', tag); continue
    run_in(env,'score_emb3.py',model,layer,tag)
if not (INTERIM/'emb3_esmc.npz').exists(): run_in('esmc','score_emb3_esmc.py')
else: print('cached esmc emb3')